# Install necessary Packages

In [3]:
import numpy as np
import pandas as pd
import requests
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

## Code to read EPUB files into a plain text string

In [4]:
def extract_epub_to_corpus(epub_path):
    book = epub.read_epub(epub_path)

    title_metadata = book.get_metadata('DC', 'title')
    title = title_metadata[0][0] if title_metadata else "Unknown Title"
    print(f"Getting: {title} info...")

    corpus_text = []

    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        soup = BeautifulSoup(item.get_body_content(), 'html.parser')
        text = soup.get_text(separator=' ', strip=True)
        corpus_text.append(text)

    full_corpus = ' '.join(corpus_text)

    return full_corpus


In [5]:
thewayofkings_path = "/kaggle/input/datasets/masonearp/sanderson-example-text/1_The_Way_of_Kings_B_Sanderson.epub"

In [6]:
thewayofkings = extract_epub_to_corpus(thewayofkings_path)

Getting: Way of Kings info...


## Length of First Novel in Corpus

In [7]:
print(len(thewayofkings))

2204860


## List of novels from BrandonSanderon to also read parse into the Corpus

In [8]:
def get_wiki_text(url):
    myheaders = {'User-Agent': 'TextasDataFinalProj/1.0 (atx6qu@virginia.edu)'}
    response = requests.get(url, headers=myheaders)
    if response.status_code == 200:
        print(f"Successfully connected to {url}")
        return response.text
    else:
        print(f"Failed. Status code: {response.status_code}")
        return None

In [9]:
def get_titles(html_string):
    soup = BeautifulSoup(html_string, 'html.parser')
    section_ids = ['Standalones', 'Mistborn', 'The_Stormlight_Archive', 'Hoid’s_Travails']
    cosmere_novels={}

    for sec_id in section_ids:
        header = soup.find('h4', id=sec_id)
        if not header:
            print(f"Warning: Could not find header for {sec_id}")
            continue
        header_div = header.find_parent('div', class_='mw-heading')
        table = header_div.find_next('table', class_='wikitable')

        if not table:
            print(f"Warning: Could not find a table for {sec_id}")
            continue

        books=[]

        for row in table.find_all('tr'):
            title_cell = row.find('th', scope='row')
            
            if title_cell:
                title = title_cell.get_text(strip = True)
                books.append(title)

        tidy_sections = sec_id.replace('_', ' ')
        cosmere_novels[tidy_sections] = books

    return cosmere_novels

In [10]:
#Website to pull from
website = "https://en.wikipedia.org/wiki/Brandon_Sanderson_bibliography"

raw_html = get_wiki_text(website)

Successfully connected to https://en.wikipedia.org/wiki/Brandon_Sanderson_bibliography


In [11]:
raw_html[:20]

'<!DOCTYPE html>\n<htm'

In [12]:
novel_dict = get_titles(raw_html)
num_novels = 0
for section, titles in novel_dict.items():
    print(f"\n{section}:")
    for title in titles:
        num_novels += 1
        print(f" - {title}")
print(f"\nTotal Novels in the Cosmere: {num_novels}")


Standalones:
 - Elantris
 - Warbreaker
 - The Sunlit Man
 - Isles of the Emberdark

Mistborn:
 - The Final Empire
 - The Well of Ascension
 - The Hero of Ages
 - The Alloy of Law
 - Shadows of Self
 - The Bands of Mourning
 - The Lost Metal

The Stormlight Archive:
 - The Way of Kings
 - Words of Radiance
 - Oathbringer
 - Rhythm of War
 - Wind and Truth

Hoid’s Travails:
 - Tress of the Emerald Sea
 - Yumi and the Nightmare Painter
 - The Fires of December

Total Novels in the Cosmere: 19


## Average Length of Novels

In [13]:
# so far this question is unknown as the other documents will need to be downloaded

## OHCO structure

-Book -Chapter -Paragraph -Terms